# Keyword analysis

In [4]:
import sys

if ".." not in sys.path:
    sys.path.append("..")

import html

import great_tables as gt
import numpy as np
import polars as pl
from cytoolz import valmap
from scipy.stats import chi2_contingency, fisher_exact

import polars_corpus as plc

In [5]:
bnc = pl.read_parquet("bnc.parquet")
speakers = pl.read_parquet("bnc-speakers.parquet")

speakers = speakers.filter(pl.col("sex").is_in(["m", "f"]))
speakers.group_by("sex").len()

bnc = (
    bnc.filter(pl.col("text_type") == "CONVRSN")
    .join(speakers, on="speaker_id", how="inner")
    .with_columns(pl.col("token").str.to_lowercase().alias("norm"))
)

In [6]:
table = (
    bnc.group_by((pl.col("norm") == "husband").alias("is_husband"), "sex")
    .agg(pl.col("norm").len().alias("count"))
    .with_columns(rel_freq=pl.col("count") / pl.col("count").sum().over("sex"))
    .sort(by=["sex", "is_husband"])
)
table

is_husband,sex,count,rel_freq
bool,str,u32,f64
false,"""f""",2745317,0.999918
true,"""f""",225,0.000082
false,"""m""",1784664,0.999967
true,"""m""",59,0.000033


In [7]:
table = np.array(table["count"]).reshape(2, 2)
table

array([[2745317,     225],
       [1784664,      59]], dtype=uint32)

In [8]:
fisher_exact(table)

SignificanceResult(statistic=np.float64(0.4033717968449212), pvalue=np.float64(3.008076355152558e-11))

In [9]:
chi2_contingency(table)

Chi2ContingencyResult(statistic=np.float64(40.47101710487801), pvalue=np.float64(1.9955429957285215e-10), dof=1, expected_freq=array([[2.74536988e+06, 1.72116626e+02],
       [1.78461112e+06, 1.11883374e+02]]))

In [10]:
np.sqrt(chi2_contingency(table).statistic / table.sum())

np.float64(0.0029888922299691526)

In [11]:
table = (
    bnc.group_by((pl.col("norm") == "wife").alias("is_wife"), "sex")
    .agg(pl.col("norm").len().alias("count"))
    .with_columns(rel_freq=pl.col("count") / pl.col("count").sum().over("sex"))
    .sort(by=["sex", "is_wife"])
)
table = np.array(table["count"]).reshape(2, 2)
table

array([[2745393,     149],
       [1784587,     136]], dtype=uint32)

In [12]:
chi2_contingency(table)

Chi2ContingencyResult(statistic=np.float64(7.926018721265425), pvalue=np.float64(0.004872890089612014), dof=1, expected_freq=array([[2.74536928e+06, 1.72722671e+02],
       [1.78461072e+06, 1.12277329e+02]]))

In [13]:
np.sqrt(chi2_contingency(table).statistic / table.sum())

np.float64(0.0013227133699140585)

In [14]:
np.sqrt(chi2_contingency(table).statistic / table.sum())

np.float64(0.0013227133699140585)

In [15]:
fisher_exact(table)

SignificanceResult(statistic=np.float64(1.404169181504793), pvalue=np.float64(0.004360351139967793))

In [16]:
table = (
    bnc.group_by((pl.col("norm") == "is").alias("is_is"), "sex")
    .agg(pl.col("norm").len().alias("count"))
    .with_columns(rel_freq=pl.col("count") / pl.col("count").sum().over("sex"))
    .sort(by=["sex", "is_is"])
)
table = np.array(table["count"]).reshape(2, 2)
table

array([[2728205,   17337],
       [1771446,   13277]], dtype=uint32)

In [17]:
chi2_contingency(table)

Chi2ContingencyResult(statistic=np.float64(203.65773353907946), pvalue=np.float64(3.3240516542754505e-46), dof=1, expected_freq=array([[2726988.55493928,   18553.44506072],
       [1772662.44506072,   12060.55493928]]))

In [18]:
np.sqrt(chi2_contingency(table).statistic / table.sum())

np.float64(0.006704843567080365)

In [19]:
fisher_exact(table)

SignificanceResult(statistic=np.float64(1.1794379252713856), pvalue=np.float64(8.788243292335871e-46))

-----

Stefanowitsch (2020), pp. 378–380

In [20]:
freq_table = plc.crosstab(bnc, pl.col("norm").str.to_lowercase(), "sex")

In [32]:
ll = freq_table.with_columns(LL=plc.loglik("f12", "f1", "f2", "n")).sort(by="LL", descending=True)

In [33]:
lex = ll.select("norm", "sex", "f12").pivot(on="sex", index="norm")

male = (
    ll.filter(
        pl.col("sex") == "m",
        pl.col("f12") > pl.col("f1") * (pl.col("f2") / pl.col("n")),
    )
    .select("norm", "LL")
    .join(lex, on="norm", how="left")
    .head(25)
    .select(pl.all().name.suffix("_m"))
)

female = (
    ll.filter(
        pl.col("sex") == "f",
        pl.col("f12") > pl.col("f1") * (pl.col("f2") / pl.col("n")),
    )
    .select("norm", "LL")
    .join(lex, on="norm", how="left")
    .head(25)
)

tbl = (
    pl.concat([male, female], how="horizontal")
    .with_columns(pl.lit("").alias("spacer"))
    .select("norm_m", "f_m", "m_m", "LL_m", "spacer", "norm", "f", "m", "LL")
    .style.fmt_number(["LL_m", "LL"], decimals=2)
    .fmt_integer(["f_m", "m_m", "f", "m"], use_seps=True)
    .fmt(html.escape, columns=["norm_m", "norm"])
    .tab_spanner("MALE", ["norm_m", "f_m", "m_m", "LL_m"])
    .tab_spanner("FEMALE", ["norm", "f", "m", "LL"])
    .cols_label(
        {
            "norm_m": "word",
            "LL_m": "LL",
            "norm": "word",
            "spacer": "",
            "f_m": "f freq",
            "m_m": "m freq",
            "f": "f freq",
            "m": "m freq",
        }
    )
    .tab_style(
        style=gt.style.css("width:50px"), locations=gt.loc.body(columns=["spacer"])
    )
    .opt_row_striping()
    .opt_vertical_padding(0.6)
)

tbl.save("LL")

GT(_tbl_data=shape: (25, 9)
┌─────────┬────────┬───────┬─────────────┬───┬──────────┬───────┬───────┬────────────┐
│ norm_m  ┆ f_m    ┆ m_m   ┆ LL_m        ┆ … ┆ norm     ┆ f     ┆ m     ┆ LL         │
│ ---     ┆ ---    ┆ ---   ┆ ---         ┆   ┆ ---      ┆ ---   ┆ ---   ┆ ---        │
│ str     ┆ u32    ┆ u32   ┆ f64         ┆   ┆ str      ┆ u32   ┆ u32   ┆ f64        │
╞═════════╪════════╪═══════╪═════════════╪═══╪══════════╪═══════╪═══════╪════════════╡
│ fucking ┆ 326    ┆ 1383  ┆ 1237.920581 ┆ … ┆ she      ┆ 22807 ┆ 7037  ┆ 3373.99179 │
│ er      ┆ 9337   ┆ 9415  ┆ 900.794048  ┆ … ┆ her      ┆ 7306  ┆ 2313  ┆ 1017.03576 │
│ the     ┆ 57367  ┆ 43385 ┆ 574.376875  ┆ … ┆ said     ┆ 12375 ┆ 4911  ┆ 915.450508 │
│ ,       ┆ 119150 ┆ 84406 ┆ 380.46747   ┆ … ┆ n't      ┆ 44380 ┆ 24221 ┆ 494.184219 │
│ yeah    ┆ 28793  ┆ 21888 ┆ 305.64689   ┆ … ┆ i        ┆ 93330 ┆ 54825 ┆ 369.237042 │
│ …       ┆ …      ┆ …     ┆ …           ┆ … ┆ …        ┆ …     ┆ …     ┆ …          │
│ four    ┆ 2284   ┆ 2116  ┆ 136.700463  ┆ … ┆ jonathan ┆ 250   ┆ 31    ┆ 113.043248 │
│ c       ┆ 358    ┆ 515   ┆ 136.218025  ┆ … ┆ think    ┆ 8985  ┆ 4862  ┆ 108.247699 │
│ number  ┆ 464    ┆ 605   ┆ 128.614672  ┆ … ┆ were     ┆ 6159  ┆ 3225  ┆ 101.291257 │
│ mate    ┆ 128    ┆ 266   ┆ 126.959068  ┆ … ┆ school   ┆ 1273  ┆ 496   ┆ 99.992118  │
│ sir     ┆ 78     ┆ 204   ┆ 125.596052  ┆ … ┆ amy      ┆ 168   ┆ 14    ┆ 95.642077  │
└─────────┴────────┴───────┴─────────────┴───┴──────────┴───────┴───────┴────────────┘, _body=<great_tables._gt_data.Body object at 0x333546030>, _boxhead=Boxhead([ColInfo(var='norm_m', type=<ColInfoTypeEnum.default: 1>, column_label='word', column_align='left', column_width=None), ColInfo(var='f_m', type=<ColInfoTypeEnum.default: 1>, column_label='f freq', column_align='right', column_width=None), ColInfo(var='m_m', type=<ColInfoTypeEnum.default: 1>, column_label='m freq', column_align='right', column_width=None), ColInfo(var='LL_m', type=<ColInfoTypeEnum.default: 1>, column_label='LL', column_align='right', column_width=None), ColInfo(var='spacer', type=<ColInfoTypeEnum.default: 1>, column_label='', column_align='left', column_width=None), ColInfo(var='norm', type=<ColInfoTypeEnum.default: 1>, column_label='word', column_align='left', column_width=None), ColInfo(var='f', type=<ColInfoTypeEnum.default: 1>, column_label='f freq', column_align='right', column_width=None), ColInfo(var='m', type=<ColInfoTypeEnum.default: 1>, column_label='m freq', column_align='right', column_width=None), ColInfo(var='LL', type=<ColInfoTypeEnum.default: 1>, column_label='LL', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x33348dd10>, _spanners=Spanners([SpannerInfo(spanner_id='MALE', spanner_level=0, spanner_label='MALE', spanner_units=None, spanner_pattern=None, vars=['norm_m', 'f_m', 'm_m', 'LL_m'], built=None), SpannerInfo(spanner_id='FEMALE', spanner_level=0, spanner_label='FEMALE', spanner_units=None, spanner_pattern=None, vars=['norm', 'f', 'm', 'LL'], built=None)]), _heading=Heading(title=None, subtitle=None, preheader=None), _stubhead=None, _source_notes=[], _footnotes=[], _styles=[StyleInfo(locname=LocBody(columns=['spacer'], rows=None, mask=None), grpname=None, colname='spacer', rownum=0, colnum=None, styles=[CellStyleCss(rule='width:50px')]), StyleInfo(locname=LocBody(columns=['spacer'], rows=None, mask=None), grpname=None, colname='spacer', rownum=1, colnum=None, styles=[CellStyleCss(rule='width:50px')]), StyleInfo(locname=LocBody(columns=['spacer'], rows=None, mask=None), grpname=None, colname='spacer', rownum=2, colnum=None, styles=[CellStyleCss(rule='width:50px')]), StyleInfo(locname=LocBody(columns=['spacer'], rows=None, mask=None), grpname=None, colname='spacer', rownum=3, colnum=None, styles=[CellStyleCss(rule='width:50px')]), StyleInfo(locname=LocBody(columns=['spacer'], rows=None, mask=None), grpname=None, colname='spacer', rownum=4, colnum=None, styles=[CellStyleCss(rule='width:50px')]), Sty

Lijffijt et al. (2016)

In [34]:
n_m = bnc.filter(pl.col("sex") == "m").n_unique("file_id")
n_f = bnc.filter(pl.col("sex") == "f").n_unique("file_id")

ttest = (
    bnc.group_by("file_id", "sex", "norm")
    .agg(pl.len().alias("freq"))
    .with_columns(n=pl.sum("freq").over(["file_id", "sex"]))
    .with_columns(
        (pl.col("freq") / pl.sum("freq").over(["file_id", "sex"])).alias("rel_freq")
    )
    .group_by("sex", "norm")
    .agg(s=pl.col("rel_freq").sum(), ss=(pl.col("rel_freq") * pl.col("rel_freq")).sum())
    .pivot(on="sex", index=["norm"])
    .drop_nulls()
    .with_columns(n_m=n_m, n_f=n_f)
    .with_columns(plc.welchs_t_from_stats("s_m", "ss_m", "n_m", "s_f", "ss_f", "n_f"))
    .unnest("t_test")
    .sort(by="pval")
)

In [35]:
ttest

norm,s_m,s_f,ss_m,ss_f,n_m,n_f,stat,pval,df
str,f64,f64,f64,f64,i32,i32,f64,f64,f64
"""she""",0.545015,1.119597,0.003714,0.014005,136,137,-7.101732,1.7800e-11,214.776578
"""her""",0.209282,0.349547,0.000694,0.001284,136,137,-4.983332,0.000001,270.939159
"""to""",1.735217,2.04982,0.023966,0.032486,136,137,-4.964395,0.000001,270.949133
"""lovely""",0.020315,0.052409,0.000013,0.000056,136,137,-4.706042,0.000005,203.745318
"""er""",0.696861,0.448582,0.006647,0.002373,136,137,3.978775,0.000096,207.297668
…,…,…,…,…,…,…,…,…,…
"""ye""",0.003106,0.003129,5.8269e-7,5.4458e-7,136,137,-0.000052,0.999958,270.317282
"""ginny""",0.000098,0.000099,6.6429e-9,9.7032e-9,136,137,0.000031,0.999975,262.674795
"""coverage""",0.000255,0.000256,3.4085e-8,4.4878e-8,136,137,0.000031,0.999975,266.671008


In [36]:
male = (
    ttest.filter(pl.col("stat") > 0)
    .select("norm", "stat", "pval")
    .join(lex, on="norm", how="left")
    .head(25)
    .select(pl.all().name.suffix("_m"))
)

male

female = (
    ttest.filter(pl.col("stat") < 0)
    .select("norm", "stat", "pval")
    .join(lex, on="norm", how="left")
    .head(25)
)

tbl = (
    pl.concat([male, female], how="horizontal")
    .with_columns(pl.lit("").alias("spacer"))
    .select(
        "norm_m",
        "f_m",
        "m_m",
        "stat_m",
        "pval_m",
        "spacer",
        "norm",
        "f",
        "m",
        "stat",
        "pval",
    )
    .style.fmt_number(["stat_m", "stat"], decimals=2)
    .fmt_number(["pval_m", "pval"], decimals=4)
    .fmt_integer(["f_m", "m_m", "f", "m"], use_seps=True)
    .fmt(html.escape, columns=["norm_m", "norm"])
    .tab_spanner("MALE", ["norm_m", "f_m", "m_m", "stat_m", "pval_m"])
    .tab_spanner("FEMALE", ["norm", "f", "m", "stat", "pval"])
    .cols_label(
        {
            "norm_m": "word",
            "stat_m": "t",
            "pval_m": "p",
            "stat": "t",
            "pval": "p",
            "norm": "word",
            "spacer": "",
            "f_m": "f freq",
            "m_m": "m freq",
            "f": "f freq",
            "m": "m freq",
        }
    )
    .tab_style(
        style=gt.style.css("width:50px"), locations=gt.loc.body(columns=["spacer"])
    )
    .opt_row_striping()
    .opt_vertical_padding(0.6)
)

tbl.save("ttest")

GT(_tbl_data=shape: (25, 11)
┌───────────┬──────┬──────┬──────────┬───┬───────┬───────┬───────────┬────────────┐
│ norm_m    ┆ f_m  ┆ m_m  ┆ stat_m   ┆ … ┆ f     ┆ m     ┆ stat      ┆ pval       │
│ ---       ┆ ---  ┆ ---  ┆ ---      ┆   ┆ ---   ┆ ---   ┆ ---       ┆ ---        │
│ str       ┆ u32  ┆ u32  ┆ f64      ┆   ┆ u32   ┆ u32   ┆ f64       ┆ f64        │
╞═══════════╪══════╪══════╪══════════╪═══╪═══════╪═══════╪═══════════╪════════════╡
│ er        ┆ 9337 ┆ 9415 ┆ 3.978775 ┆ … ┆ 22807 ┆ 7037  ┆ -7.101732 ┆ 1.7800e-11 │
│ which     ┆ 1498 ┆ 1417 ┆ 3.686964 ┆ … ┆ 7306  ┆ 2313  ┆ -4.983332 ┆ 0.000001   │
│ c         ┆ 358  ┆ 515  ┆ 3.65739  ┆ … ┆ 40934 ┆ 23693 ┆ -4.964395 ┆ 0.000001   │
│ quid      ┆ 337  ┆ 476  ┆ 3.537433 ┆ … ┆ 1217  ┆ 406   ┆ -4.706042 ┆ 0.000005   │
│ ah        ┆ 2606 ┆ 2374 ┆ 3.418032 ┆ … ┆ 571   ┆ 174   ┆ -3.961189 ┆ 0.000098   │
│ …         ┆ …    ┆ …    ┆ …        ┆ … ┆ …     ┆ …     ┆ …         ┆ …          │
│ inch      ┆ 39   ┆ 74   ┆ 2.883845 ┆ … ┆ 23472 ┆ 13236 ┆ -3.293112 ┆ 0.001126   │
│ seriously ┆ 51   ┆ 74   ┆ 2.858979 ┆ … ┆ 59    ┆ 13    ┆ -3.272286 ┆ 0.001241   │
│ forty     ┆ 562  ┆ 623  ┆ 2.820484 ┆ … ┆ 3523  ┆ 1545  ┆ -3.234393 ┆ 0.001388   │
│ brake     ┆ 13   ┆ 38   ┆ 2.822812 ┆ … ┆ 1027  ┆ 552   ┆ -3.216272 ┆ 0.001459   │
│ four      ┆ 2284 ┆ 2116 ┆ 2.774288 ┆ … ┆ 72    ┆ 20    ┆ -3.126483 ┆ 0.002078   │
└───────────┴──────┴──────┴──────────┴───┴───────┴───────┴───────────┴────────────┘, _body=<great_tables._gt_data.Body object at 0x328769d90>, _boxhead=Boxhead([ColInfo(var='norm_m', type=<ColInfoTypeEnum.default: 1>, column_label='word', column_align='left', column_width=None), ColInfo(var='f_m', type=<ColInfoTypeEnum.default: 1>, column_label='f freq', column_align='right', column_width=None), ColInfo(var='m_m', type=<ColInfoTypeEnum.default: 1>, column_label='m freq', column_align='right', column_width=None), ColInfo(var='stat_m', type=<ColInfoTypeEnum.default: 1>, column_label='t', column_align='right', column_width=None), ColInfo(var='pval_m', type=<ColInfoTypeEnum.default: 1>, column_label='p', column_align='right', column_width=None), ColInfo(var='spacer', type=<ColInfoTypeEnum.default: 1>, column_label='', column_align='left', column_width=None), ColInfo(var='norm', type=<ColInfoTypeEnum.default: 1>, column_label='word', column_align='left', column_width=None), ColInfo(var='f', type=<ColInfoTypeEnum.default: 1>, column_label='f freq', column_align='right', column_width=None), ColInfo(var='m', type=<ColInfoTypeEnum.default: 1>, column_label='m freq', column_align='right', column_width=None), ColInfo(var='stat', type=<ColInfoTypeEnum.default: 1>, column_label='t', column_align='right', column_width=None), ColInfo(var='pval', type=<ColInfoTypeEnum.default: 1>, column_label='p', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x3287b1480>, _spanners=Spanners([SpannerInfo(spanner_id='MALE', spanner_level=0, spanner_label='MALE', spanner_units=None, spanner_pattern=None, vars=['norm_m', 'f_m', 'm_m', 'stat_m', 'pval_m'], built=None), SpannerInfo(spanner_id='FEMALE', spanner_level=0, spanner_label='FEMALE', spanner_units=None, spanner_pattern=None, vars=['norm', 'f', 'm', 'stat', 'pval'], built=None)]), _heading=Heading(title=None, subtitle=None, preheader=None), _stubhead=None, _source_notes=[], _footnotes=[], _styles=[StyleInfo(locname=LocBody(columns=['spacer'], rows=None, mask=None), grpname=None, colname='spacer', rownum=0, colnum=None, styles=[CellStyleCss(rule='width:50px')]), StyleInfo(locname=LocBody(columns=['spacer'], rows=None, mask=None), grpname=None, colname='spacer', rownum=1, colnum=None, styles=[CellStyleCss(rule='width:50px')]), StyleInfo(locname=LocBody(columns=['spacer'], rows=None, mask=None), grpname=None, colname='spacer', rownum=2, colnum=None, styles=[CellStyleCss(rule='width:50px')]), StyleInfo(locname=LocBody(columns=['spacer'], rows=None, mask=None), grpname=None, colname='spacer', rownum=3, colnum=None, styles=

Word shift graphs

In [ ]:
female_freq = valmap(
    lambda x: x[0],
    lex.select("norm", "f").drop_nulls().rows_by_key(key="norm", unique=True),
)
male_freq = valmap(
    lambda x: x[0],
    lex.select("norm", "m").drop_nulls().rows_by_key(key="norm", unique=True),
)

In [ ]:
##prop = sh.ProportionShift(female_freq, male_freq)
# p#rop.get_shift_graph()